# 보호색 동물: Colab CPU 데이터 준비 → GPU 첫 학습

CPU에서는 ①~⑤까지 실행합니다. GPU가 연결된 뒤 ⑥~⑦을 실행합니다. 런타임 변경 시 /content가 사라질 수 있어 검출 학습 데이터만 Drive에 백업합니다. 전체 MoCA 추적 데이터는 백업에 포함하지 않습니다.

COD10K는 nobg/COD10K 공개 미러의 보호색 이미지 5,066쌍(Background/NonCAM 미포함), MoCA는 Oxford 공식 ZIP을 사용합니다. 전체 모드는 이 작업 환경에서 미실행이며, 실제 Colab 실행 중 발견되는 원본 오류는 검사 결과에 따라 수정해야 합니다.


## ① CPU 연결 상태와 공간 확인

런타임 → 런타임 유형 변경 → CPU. 전체 준비 권장 여유 공간은 40GiB 이상입니다. GPU 할당 실패 시 CPU 준비만 진행하세요.


In [ ]:
import shutil, psutil, torch
GiB = 1024**3
print("GPU:", torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0), torch.cuda.mem_get_info(0))
print("RAM 여유 GiB:", round(psutil.virtual_memory().available/GiB, 1))
free = shutil.disk_usage("/content").free/GiB
print("디스크 여유 GiB:", round(free, 1))
print("전체 데이터 공간 기준:", "충족" if free >= 40 else "부족 — 표본 모드 또는 추가 공간 필요")


## ② 코드 준비

처음 실행 시 저장소를 복제합니다. 기존 폴더가 있다면 브랜치 상태를 확인하고, 다른 작업이 있는 폴더를 강제로 바꾸지 않습니다.


In [ ]:
from pathlib import Path
import subprocess
PROJECT = Path("/content/AI_Project_Level_3-main")
BRANCH = "feature/camouflage-first-training"
if not PROJECT.exists():
    subprocess.run(["git", "clone", "--branch", BRANCH, "https://github.com/kumohyunho/AI_Project_Level_3-main.git", str(PROJECT)], check=True)
%cd /content/AI_Project_Level_3-main
print(subprocess.check_output(["git", "branch", "--show-current"], text=True))
assert (PROJECT/"fetch_camouflage_data.py").is_file(), "필요한 코드가 없습니다. 브랜치를 확인하세요."
%pip install -r requirements.txt -r requirements-camouflage-data.txt


## ③ 원본 다운로드 (시간이 걸림)

FULL=True는 전체 보호색 이미지와 전체 MoCA를 받습니다. MoCA ZIP만 약 9.9GB이며 인터넷 속도에 따라 오래 걸립니다. 작은 실행 점검을 원하면 FULL=False로 바꾸세요. 기존 원본 폴더를 덮어쓰지 않습니다. 표본은 전체 대표 표본이 아니며 성능 결론에 사용하지 않습니다.


In [ ]:
FULL = True
RAW = PROJECT/"datasets/raw/camouflage-colab"
cmd = ["python", "-u", "fetch_camouflage_data.py", "--output", str(RAW)]
if FULL:
    cmd.append("--full")
if (RAW/"acquisition.json").is_file():
    print("기존 원본 유지:", RAW)
else:
    subprocess.run(cmd, check=True)

import json
acquisition = json.loads((RAW/"acquisition.json").read_text())
print("실제 확보 범위:", acquisition["scope"])
assert FULL == acquisition["full"], "기존 폴더와 요청 범위가 다릅니다. RAW 경로를 새 이름으로 바꾸세요."


## ④ 전처리와 #8 검사

MoCA 원본의 motion 번호는 개체 ID가 아니므로 추적 ID는 비워 둡니다. ID 미확인 경고가 있어도 검출 학습은 가능하지만 추적 ID 성능 평가는 할 수 없습니다. 이미지에서 FPS를 확인할 수 없어 24는 검출 학습용 임시값이며 시간 성능 분석에 쓰지 않습니다.


In [ ]:
import json
DATA = PROJECT/"datasets/camouflage"
REVIEW = PROJECT/"results/camouflage/colab-data-check"
if (DATA/"splits.json").is_file():
    existing = json.loads((DATA/"splits.json").read_text())
    assert all(Path(r["original"]).is_relative_to(RAW) for r in existing["records"]), "기존 데이터가 다른 원본에서 생성됐습니다. DATA와 REVIEW를 새 경로로 지정하세요."
if not (DATA/"splits.json").is_file():
    subprocess.run(["python", "camouflage_dataset.py", "--cod10k", str(RAW/"COD10K-v3"),
                    "--moca-frames", str(RAW/"MoCA/frames"), "--moca-csv", str(RAW/"MoCA/boxes.csv"),
                    "--fps", "24", "--output", str(DATA)], check=True)
if not (REVIEW/"report.json").is_file():
    subprocess.run(["python", "inspect_camouflage_data.py", str(DATA), "--check-hashes", "--samples", "18",
                    "--output", str(REVIEW)], check=True)
report = json.loads((REVIEW/"report.json").read_text())
print("오류:", report["errors"], "경고:", report["warnings"])
print(json.loads((DATA/"splits.json").read_text())["counts"])
assert report["errors"] == 0, "구조 오류를 먼저 수정하세요. 자동으로 무시하지 않습니다."
from IPython.display import display, Image as DisplayImage
for preview in report["previews"]:
    print(preview["image"], "선언된 ID:", preview["declared_ids"])
    display(DisplayImage(filename=str(REVIEW/preview["preview"])))


## ⑤ Drive에 검출 학습 데이터 백업

Drive의 남은 용량은 Google Drive 저장용량 화면에서 먼저 확인하세요. /content 디스크와 Drive 할당량은 별개입니다. images/labels와 학습 명세만 백업하므로 추적 시퀀스는 포함하지 않습니다. 기존 백업 파일은 덮어쓰지 않습니다.


In [ ]:
from google.colab import drive
import tarfile
drive.mount("/content/drive")
DRIVE = Path("/content/drive/MyDrive/camouflage_learning")
DRIVE.mkdir(parents=True, exist_ok=True)
ARCHIVE = DRIVE/"detection_data.tar.gz"
if ARCHIVE.exists():
    print("기존 백업 유지. 다른 데이터를 백업하려면 ARCHIVE 이름을 바꾸세요:", ARCHIVE)
else:
    assert report["errors"] == 0
    partial = ARCHIVE.with_suffix(".partial")
    assert not partial.exists(), "중단된 백업이 있습니다. Drive에서 확인 후 이름을 바꾸거나 정리하세요."
    with tarfile.open(partial, "w:gz") as tar:
        for name in ("images", "labels", "dataset.yaml", "splits.json"):
            tar.add(DATA/name, arcname=name)
    partial.rename(ARCHIVE)
    shutil.copy2(REVIEW/"report.json", DRIVE/"data_check_report.json")
    shutil.copy2(RAW/"acquisition.json", DRIVE/"acquisition.json")
    print("백업 완료:", ARCHIVE)


## ⑥ GPU가 연결된 뒤 데이터 복원

런타임 → 런타임 유형 변경 → T4 GPU → 저장. 계속 연결 중이거나 할당 실패이면 여기부터 실행하지 마세요. 런타임이 바뀌었다면 ①·②를 다시 실행하고 아래 셀로 백업을 복원합니다. ③·④ 전체 다운로드를 다시 실행할 필요는 없습니다.


In [ ]:
import torch, tarfile, shutil
from tempfile import TemporaryDirectory
from google.colab import drive
assert torch.cuda.is_available(), "GPU가 없습니다. CPU 준비만 완료하고 GPU 할당 후 재시도하세요."
print("연결 GPU:", torch.cuda.get_device_name(0))
drive.mount("/content/drive")
DRIVE = Path("/content/drive/MyDrive/camouflage_learning")
DATA = PROJECT/"datasets/camouflage"
if not DATA.exists():
    DATA.parent.mkdir(parents=True, exist_ok=True)
    with TemporaryDirectory(dir=DATA.parent) as tmp:
        staging = Path(tmp)/"restored"
        staging.mkdir()
        with tarfile.open(DRIVE/"detection_data.tar.gz", "r:gz") as tar:
            tar.extractall(staging, filter="data")
        staging.rename(DATA)
assert (DATA/"dataset.yaml").is_file()
print("검출 학습 데이터 준비 완료. 추적 시퀀스는 별도 원본 준비가 필요합니다.")


## ⑦ 첫 GPU 학습

3에폭은 학습 경로 확인용입니다. 충분한 성능이나 수렴을 보장하지 않습니다. GPU 메모리 부족이면 batch를 4 또는 2로 낮추세요. 체크포인트를 Drive에 기록하며, 기존 camouflage.pt/json이 있으면 함께 백업한 뒤 설치합니다. 기존 모델을 자동 삭제하지 않습니다.


In [ ]:
assert torch.cuda.is_available(), "GPU 연결 후 실행하세요."
subprocess.run(["python", "train_camouflage.py", "--data", str(DATA/"dataset.yaml"),
                "--epochs", "3", "--imgsz", "640", "--batch", "8", "--device", "0",
                "--runs", str(DRIVE/"training_runs"), "--install"], check=True)
(DRIVE/"models").mkdir(exist_ok=True)
for name in ("camouflage.pt", "camouflage.json"):
    shutil.copy2(PROJECT/"models"/name, DRIVE/"models"/name)
print("가중치·설정 저장:", DRIVE/"models")
